# 🧑‍⚕️ Mixture-of-Experts

Many experts, but a router activates only a few per token.

> ▶︎ In Colab: **Runtime → Run all** — this notebook runs top to bottom with no setup.

In [ ]:
%pip install -q numpy

In [ ]:
import numpy as np

np.random.seed(0)
n_experts, top_k = 4, 2
router = np.random.randn(8, n_experts)        # 8 tokens, scores per expert

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

gates = softmax(router)
for t, g in enumerate(gates):
    chosen = np.argsort(g)[-top_k:][::-1]
    print(f'token {t} -> experts {chosen.tolist()} (the other {n_experts - top_k} stay idle)')

## Try it

Load balancing is the hard part. Count how often each expert fires — if one hogs every token, training breaks.

In [ ]:
from collections import Counter
counts = Counter()
for g in gates:
    for e in np.argsort(g)[-top_k:]:
        counts[int(e)] += 1
print('expert load:', dict(sorted(counts.items())))
print('Balanced routing keeps every expert busy; skew wastes capacity.')

## Takeaway

- Huge total parameters, small compute per token.
- The router (and its load balancing) is the whole game.

## 🚀 Your move

Build: code a toy router that sends each input vector to its top-2 of 4 experts by a learned score, and print which expert fires when. Then make one expert greedy and watch load-balancing break.